In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    round, avg, datediff
)
from datetime import date

# ── Load all gold tables needed ────────────────────────────
cost_pred    = spark.table("llm_pulse_dev.gold.cost_predictions")
quality_alrt = spark.table("llm_pulse_dev.gold.quality_alerts")
daily_cost   = spark.table("llm_pulse_dev.gold.daily_cost_by_team")
quality_score= spark.table("llm_pulse_dev.gold.quality_score_daily")

# ── Build cost alert flags ─────────────────────────────────
# Flag predictions that exceed 120% of recent average spend
team_avg = (
    daily_cost
    .groupBy("team")
    .agg(round(avg("total_cost_usd"), 6).alias("historical_avg_cost"))
)

cost_alerts = (
    cost_pred
    .join(team_avg, on="team", how="left")
    .withColumn("cost_alert_flag",
        when(col("predicted_cost_usd") > col("historical_avg_cost") * 1.2,
             True).otherwise(False))
    .withColumn("cost_alert_severity",
        when(col("predicted_cost_usd") > col("historical_avg_cost") * 1.5,
             "critical")
        .when(col("predicted_cost_usd") > col("historical_avg_cost") * 1.2,
             "warning")
        .otherwise("normal"))
    .withColumn("alert_type", lit("cost_spike_predicted"))
    .select(
        col("prediction_date").alias("alert_date"),
        col("team"),
        lit(None).cast("string").alias("model"),
        col("predicted_cost_usd").alias("metric_value"),
        col("historical_avg_cost").alias("baseline_value"),
        col("cost_alert_flag").alias("is_alert"),
        col("cost_alert_severity").alias("severity"),
        col("alert_type"),
        lit("cost_forecasting_model").alias("detected_by")
    )
)

# ── Build quality alert flags ──────────────────────────────
quality_alerts_formatted = (
    quality_alrt
    .withColumn("baseline_value", lit(88.0))  # expected ~88% quality
    .withColumn("is_alert", lit(True))
    .withColumn("alert_type", lit("quality_drop_detected"))
    .withColumn("detected_by", lit("anomaly_detection_model"))
    .select(
        col("feedback_date").alias("alert_date"),
        lit(None).cast("string").alias("team"),
        col("model"),
        col("quality_score_pct").alias("metric_value"),
        col("baseline_value"),
        col("is_alert"),
        col("alert_severity").alias("severity"),
        col("alert_type"),
        col("detected_by")
    )
)

# ── Union cost and quality alerts ──────────────────────────
alert_summary = (
    cost_alerts
    .union(quality_alerts_formatted)
    .withColumn("created_at", current_timestamp())
    .orderBy("alert_date", "severity")
)

# ── Save to gold schema ────────────────────────────────────
(alert_summary
    .write.option('mergeSchema', 'true')
    .mode("overwrite")
    .saveAsTable("llm_pulse_dev.gold.alert_summary"))

print(f"Alert summary saved!")
alert_summary.groupBy("alert_type", "severity").count().show()

In [0]:
%sql
SHOW TABLES IN llm_pulse_dev.gold;


In [0]:
%sql
-- Full project summary — one query to see everything
SELECT
    'Total API calls processed'        AS metric, COUNT(*) AS value
    FROM llm_pulse_dev.silver.api_calls
UNION ALL
SELECT
    'Total cost tracked (USD)',          ROUND(SUM(cost_usd), 2)
    FROM llm_pulse_dev.silver.api_calls
UNION ALL
SELECT
    'Days of data',                      COUNT(DISTINCT event_date)
    FROM llm_pulse_dev.silver.api_calls
UNION ALL
SELECT
    'Cost spike alerts generated',       COUNT(*)
    FROM llm_pulse_dev.gold.alert_summary
    WHERE alert_type = 'cost_spike_predicted' AND is_alert = true
UNION ALL
SELECT
    'Quality drop alerts generated',     COUNT(*)
    FROM llm_pulse_dev.gold.alert_summary
    WHERE alert_type = 'quality_drop_detected'
UNION ALL
SELECT
    'Days of cost forecasted',           COUNT(DISTINCT prediction_date)
    FROM llm_pulse_dev.gold.cost_predictions;